# Plaka Tespit Modeli Egitimi - YOLO11n + Roboflow

Bu not defteri, arac plakasi tespiti icin YOLO11n modelini Google Colab ortaminda egitmek amaciyla hazirlandi. Egitim sonunda olusan `best.pt` dosyasi yerel projede `models/best.pt` olarak kullanilacaktir.

## 1. GPU Kontrolu

Colab menüsünden `Runtime > Change runtime type > GPU` secilmelidir.

In [ ]:
!nvidia-smi

## 2. Paket Kurulumu

In [ ]:
!pip install -q ultralytics roboflow easyocr opencv-python-headless pandas matplotlib

In [ ]:
from ultralytics import YOLO
import ultralytics
ultralytics.checks()

## 3. Roboflow Veri Setini Indirme

Kullanilacak veri seti: Roboflow Universe `License Plate Recognition v11`. API anahtari Roboflow hesabindan alinabilir. Anahtar deftere yazdirilmaz; `getpass` ile girilir.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

ROBOFLOW_API_KEY = getpass('Roboflow API key: ')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('roboflow-universe-projects').project('license-plate-recognition-rxg4e')
version = project.version(11)
dataset = version.download('yolov11')
DATA_YAML = f'{dataset.location}/data.yaml'
print(DATA_YAML)

## 4. Veri Seti Kontrolu

YOLO formatinda `images/train`, `images/valid` ve `images/test` klasorleri ile paralel `labels` klasorleri beklenir.

In [ ]:
from pathlib import Path
dataset_root = Path(dataset.location)
for split in ['train', 'valid', 'test']:
    image_count = len(list((dataset_root / split / 'images').glob('*')))
    label_count = len(list((dataset_root / split / 'labels').glob('*.txt')))
    print(split, 'images:', image_count, 'labels:', label_count)

!cat {DATA_YAML}

## 5. Google Drive Kayit Alani

Colab'in `/content` klasoru gecicidir. Baglanti koparsa egitim ciktisi silinebilir. Bu nedenle model agirliklari Google Drive'a yazdirilir.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/plaka_projesi/plate_runs'
RUN_NAME = 'yolo11n_plate'
Path(PROJECT_DIR).mkdir(parents=True, exist_ok=True)
print('Egitim ciktisi buraya kaydedilecek:', PROJECT_DIR)

## 5. YOLO11n Model Egitimi

Ilk prototip icin hafif ve hizli model olan `yolo11n.pt` secildi. GPU kopma riskini azaltmak icin epoch sayisi 30 tutuldu. `save_period=5` ayariyla her 5 epoch'ta ara checkpoint dosyasi Drive'a kaydedilir.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
train_results = model.train(
    data=DATA_YAML,
    epochs=30,
    imgsz=640,
    batch=16,
    project=PROJECT_DIR,
    name=RUN_NAME,
    patience=10,
    save_period=5,
    plots=True
)

## 6. Dogrulama ve Test

Bu hucrelerde olusan mAP degerleri rapora gercek sonuc olarak yazilacaktir.

In [ ]:
from ultralytics import YOLO

best_weights = f'{PROJECT_DIR}/{RUN_NAME}/weights/best.pt'
best_model = YOLO(best_weights)
val_metrics = best_model.val(data=DATA_YAML, split='val')
test_metrics = best_model.val(data=DATA_YAML, split='test')

print('VAL mAP50:', float(val_metrics.box.map50))
print('VAL mAP50-95:', float(val_metrics.box.map))
print('TEST mAP50:', float(test_metrics.box.map50))
print('TEST mAP50-95:', float(test_metrics.box.map))

## 7. Ornek Tahmin ve Gorsel Cikti

In [ ]:
predict_results = best_model.predict(
    source=str(dataset_root / 'test' / 'images'),
    conf=0.25,
    save=True,
    max_det=5
)
print('Tahmin sonuclari runs/detect klasorune kaydedildi.')

## 8. EasyOCR ile Bir Kirpim Uzerinde Demo

Bu bolum egitimden bagimsiz olarak OCR kutuphanesinin plaka kirpimi uzerinde nasil calistigini gosterir. Yerel projede ayni mantik `src/ocr_reader.py` icinde kullanilmistir.

In [ ]:
import cv2
import easyocr
from matplotlib import pyplot as plt

reader = easyocr.Reader(['en'], gpu=True)
sample_image = next((dataset_root / 'test' / 'images').glob('*'))
image = cv2.imread(str(sample_image))
result = best_model.predict(source=image, conf=0.25, verbose=False)[0]

if result.boxes is not None and len(result.boxes) > 0:
    box = result.boxes.xyxy[0].cpu().numpy().astype(int)
    x1, y1, x2, y2 = box
    crop = image[y1:y2, x1:x2]
    ocr_result = reader.readtext(crop, allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789')
    print(ocr_result)
    plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    plt.axis('off')
else:
    print('Ornek gorselde plaka tespit edilemedi.')

## 9. Model Agirligini Indirme

`best.pt` dosyasi Drive'a da kaydedilmistir. Yine de indirip yerel projede `models/best.pt` olarak saklayin.

In [ ]:
from google.colab import files
files.download(best_weights)
print('Drive uzerindeki model yolu:', best_weights)